# DiD-BCF — D_staggered (linearity_degree=3)

**Workstream D · staggered adoption (cohort x event-time effects)**

dynamic, cohort-varying effects (Goodman-Bacon case)

Fits DiD-BCF on the `D_staggered` scenario at `linearity_degree=3` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.9 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_staggered",
    linearity_degree=3,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_staggered_lin_3] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_staggered: 100%|██████████| 100/100 [6:14:04<00:00, 224.45s/fit]

[D_staggered_lin_3] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_staggered_lin_3.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_staggered,3,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.282667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_staggered,3,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.185333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.824834
2,staggered,D_staggered,3,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.082667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
3,staggered,D_staggered,3,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.047333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.010453
4,staggered,D_staggered,3,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.053333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.147290


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_staggered,3,200,ATT,ATT,power,-1.007635,-4.241687,0.03,...,0.799706,2.197019,1.007635,4.241687,1.0,0.50,1.077883,4.251019,0.491825,2.129323
1,staggered,D_staggered,3,200,ES,k=0,power,-0.396954,-3.721497,0.39,...,0.778007,1.205033,0.424111,3.721497,1.0,0.01,0.498585,3.726303,0.609014,2.470200
2,staggered,D_staggered,3,200,ES,k=1,power,-1.213777,-4.653846,0.01,...,0.874480,2.082147,1.213777,4.653846,1.0,0.57,1.276542,4.663988,0.528733,2.065840
3,staggered,D_staggered,3,200,ES,k=2,power,-1.447402,-4.531276,0.02,...,1.117657,3.127245,1.447402,4.531276,1.0,0.56,1.527387,4.548581,0.525915,1.989077
4,staggered,D_staggered,3,200,ES,k=3,power,-1.292978,-3.981368,0.14,...,1.538301,4.055051,1.293791,3.981368,1.0,0.54,1.410189,4.004332,0.622464,2.362435
5,staggered,D_staggered,3,200,GATT,g=4_t=4,power,-0.018624,-2.746948,0.82,...,1.424864,2.417413,0.317397,2.746948,1.0,0.00,0.392603,2.749749,0.875840,31.830176
6,staggered,D_staggered,3,200,GATT,g=4_t=5,power,-0.644744,-3.413447,0.49,...,1.435640,2.864471,0.668524,3.413447,1.0,0.00,0.769800,3.424639,0.794624,2.768115
7,staggered,D_staggered,3,200,GATT,g=4_t=6,power,-1.063774,-3.837466,0.20,...,1.462250,3.521737,1.068970,3.837466,1.0,0.04,1.193114,3.859343,0.642739,2.246745
8,staggered,D_staggered,3,200,GATT,g=4_t=7,power,-1.292978,-3.981368,0.14,...,1.538301,4.055051,1.293791,3.981368,1.0,0.54,1.410189,4.004332,0.622464,2.362435
9,staggered,D_staggered,3,200,GATT,g=5_t=5,power,-0.375639,-3.695507,0.73,...,1.579943,1.630111,0.446963,3.695507,1.0,0.16,0.544600,3.709363,0.962714,1.751678


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_staggered,3,200,corrected,100,431.6,2.153162,0.216880,1.696368,...,0.319706,0.017971,0.236935,0.039128,0.285672,0.045792,1.295628,0.127041,1.566355,0.162520
1,staggered,D_staggered,3,200,plain,100,431.6,4.645148,0.282233,4.241687,...,0.809429,0.050952,0.026703,0.022383,0.049741,0.037939,2.450204,0.197864,2.983448,0.279327


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=3)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.063198924628939,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contributi